# 🤖 FLEET × SaferPath — Hybrid SOTA Navigation Training

**Novel benchmark-setting training** that combines:
- FLEET CBF-QP formal safety guarantees
- SaferPath traversability-aware navigation (arXiv:2603.01898)
- Zone-aware CMDP-Lagrangian cost shaping
- 6 scenario types (3 SaferPath + 3 novel FLEET)

**Models trained:**
1. `FLEET-SaferPath-Hybrid` — TraversabilityCBF Transformer
2. Comparison benchmarks vs SaferPath, ViNT, NoMaD

**W&B Project:** `fleet-safe-vla`

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

import numpy as np
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Dataset — SaferPath + FLEET Scenarios

In [ ]:
from training.saferpath_hybrid_train import SaferPathDataset

dataset = SaferPathDataset(
    n_episodes=1500,
    obs_dim=48, act_dim=2,
    trav_map_size=32,
    include_saferpath_scenarios=True,
    include_fleet_scenarios=True,
)

# Scenario distribution
from collections import Counter
scenarios = Counter(ep['scenario'] for ep in dataset.episodes)
print("\n📊 Scenario distribution:")
for s, c in scenarios.most_common():
    print(f"  {s:<25s}: {c:>4d} episodes")
print(f"  {'TOTAL':<25s}: {len(dataset.episodes):>4d}")

In [ ]:
# Visualize traversability maps
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
scenario_names = SaferPathDataset.SCENARIOS

for idx, (ax, scenario) in enumerate(zip(axes.flat, scenario_names)):
    # Find an episode for this scenario
    ep = next(e for e in dataset.episodes if e['scenario'] == scenario)
    trav = ep['trav_maps'][ep['steps']//2]  # Middle timestep
    
    ax.imshow(trav, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(scenario.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Traversability Maps — SaferPath + FLEET Scenarios', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_logs/saferpath_hybrid/traversability_maps.png', dpi=150)
plt.show()
print('Saved traversability_maps.png')

## 2. Model Architecture — TraversabilityCBF Transformer

In [ ]:
from training.saferpath_hybrid_train import TraversabilityCBFPolicy

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = TraversabilityCBFPolicy(
    obs_dim=48, act_dim=2, trav_map_size=32,
    hidden_dim=512, n_layers=8, n_heads=8,
).to(device)

param_count = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n🏗️ TraversabilityCBF Policy")
print(f"  Total params:     {param_count:>12,} ({param_count/1e6:.1f}M)")
print(f"  Trainable:        {trainable:>12,}")
print(f"  Architecture:     Transformer (8L × 8H × 512D)")
print(f"  Traversability:   32×32 map encoder")
print(f"  Safety heads:     CBF barrier + Zone cost + CMDP λ")
print(f"  Action space:     2D velocity (vx, vy)")
print(f"  Device:           {device}")

In [ ]:
# Quick forward pass test
batch = dataset.sample_batch(4, horizon=16)
with torch.no_grad():
    out = model(batch['obs'].to(device), batch['trav_maps'].to(device))

print("\n✅ Forward pass shapes:")
for k, v in out.items():
    if torch.is_tensor(v):
        print(f"  {k:<20s}: {tuple(v.shape)}")
    else:
        print(f"  {k:<20s}: {v:.4f}")

## 3. 🚀 TRAIN — Full SaferPath Hybrid Run

In [ ]:
from training.saferpath_hybrid_train import train_saferpath_hybrid
from argparse import Namespace

args = Namespace(
    epochs=300,
    batch_size=64,
    lr=3e-4,
    episodes=1500,
    dry_run=False,
)

print("🚀 Starting FLEET × SaferPath hybrid training...")
print(f"   Epochs: {args.epochs} | Batch: {args.batch_size} | Episodes: {args.episodes}")
result = train_saferpath_hybrid(args)

## 4. Results Analysis

In [ ]:
import json

result_path = 'training_logs/saferpath_hybrid/result.json'
with open(result_path) as f:
    result = json.load(f)

print("\n" + "═" * 60)
print("  FLEET × SaferPath — Training Results")
print("═" * 60)
print(f"  Model:          {result['model']}")
print(f"  Parameters:     {result['parameters']:,}")
print(f"  Epochs:         {result['epochs']}")
print(f"  Training time:  {result['training_time_s']:.1f}s")
print(f"  Device:         {result['device']}")
print()
print(f"  📈 Final Loss:       {result['final_loss']:.6f}")
print(f"  ✅ Success Rate:     {result['final_success_rate']:.1%}")
print(f"  💥 Collision Rate:   {result['final_collision_rate']:.1%}")
print(f"  🛡️  SVR:             {result['final_svr']:.5f}")
print(f"  📏 SPL:              {result['final_spl']:.3f}")
print(f"  🔒 Barrier Mean:     {result['final_barrier_mean']:.4f}")
print(f"  λ (Lagrange):        {result['final_lambda']:.4f}")

In [ ]:
# Comparison table
comp = result.get('comparison_vs_saferpath', {})
print("\n" + "═" * 60)
print("  HEAD-TO-HEAD: FLEET vs SaferPath")
print("═" * 60)
print(f"  Success Δ:    {comp.get('success_delta', 0):+.1%} vs SaferPath")
print(f"  Collision Δ:  {comp.get('collision_delta', 0):+.1%} vs SaferPath")
print(f"  SPL Δ:        {comp.get('spl_delta', 0):+.3f} vs SaferPath")
print(f"  Formal CBF:   {'✅ FLEET' if comp.get('has_formal_guarantee') else '❌'}")
print(f"  SaferPath:    {'✅' if comp.get('saferpath_has_formal_guarantee') else '❌ No formal guarantee'}")
print()
print("  🏆 Novel contributions:")
for i, inn in enumerate(result.get('innovations', []), 1):
    print(f"    {i}. {inn}")

## 5. Benchmark Comparison Tables (Paper-ready)

In [ ]:
from training.saferpath_benchmark import generate_comparison, generate_markdown_table

comparison = generate_comparison()
md = generate_markdown_table(comparison)
print(md)

In [ ]:
# Updated paper benchmarks with all 10+ baselines
from training.paper_benchmarks import generate_comparison_table, save_benchmarks

md = save_benchmarks()
print(md)

## 6. Robot Registry — URDF Models

In [ ]:
from robots.registry import get_robot_registry

reg = get_robot_registry()
print(f"\n🤖 Robot Registry — {len(reg)} robots loaded\n")
print(f"{'ID':<22s} │ {'Name':<28s} │ {'DOF':>4s} │ {'Mass':>8s} │ {'Type':<12s} │ Sensors")
print("─" * 100)
for r in reg.list_robots():
    print(f"{r['id']:<22s} │ {r['display_name']:<28s} │ {r['actuated_dof']:>4d} │ "
          f"{r['total_mass_kg']:>7.2f}kg │ {r['embodiment']:<12s} │ {', '.join(r['sensors'])}")

---
## Summary

✅ **Trained** FLEET-SaferPath-Hybrid model on 6 navigation scenarios  
✅ **CBF-QP safety guarantee** — formal forward invariance (SaferPath has none)  
✅ **Novel data** — barrier-annotated traversability maps (unique contribution)  
✅ **10+ baselines** — SaferPath, ViNT, NoMaD, SafeVLA, RT-2, OpenVLA, GR00T-N1...  
✅ **4 robot URDFs** — G1, Go2, TurtleBot3, Fetch for cross-embodiment  
✅ **W&B integrated** — full metric tracking for paper figures  